# 1. 문서의 내용을 읽는다.

In [ ]:
%pip install python-docx

In [ ]:
from docx import Document

document = Document('./tax_docs/tax_with_markdown.docx')
print(f'document == {dir(document)}')
full_text = ''
for index, paragraph in enumerate(document.paragraphs):
    print(f'paragraph == {paragraph.text}')
    full_text += f'{paragraph.text}\n'

# 2.문서를 쪼갠다

In [ ]:
%pip install tiktoken

In [ ]:
import tiktoken

def split_text(full_text, chunk_size):
    encoder = tiktoken.encoding_for_model('gpt-4o')
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    text_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i:i + chunk_size]
        decoded = encoder.decode(chunk)
        text_list.append(decoded)
    return text_list

In [ ]:
chunk_list = split_text(full_text, 1500)

In [ ]:
chunk_list

# 3.문서 임베딩

In [ ]:
%pip install chromadb

In [ ]:
import chromadb

chroma_client = chromadb.Client()

In [ ]:
tax_collection = chroma_client.create_collection(name='tax_collection')

In [ ]:
import os
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')

openai_embedding = OpenAIEmbeddingFunction(model_name='text-embedding-3-large', api_key=openai_api_key)

In [ ]:
# 기존 default 임베딩으로 생성된 컬렉션 삭제 후 openai 임베딩으로 재생성
chroma_client.delete_collection(name='tax_collection')
tax_collection = chroma_client.get_or_create_collection(name='tax_collection', embedding_function=openai_embedding)

In [ ]:
id_list =[]
for index in range(len(chunk_list)):
  id_list.append(f'{index}')


In [ ]:
len(id_list)

In [ ]:
len(chunk_list)

In [ ]:
tax_collection.add(documents=chunk_list, ids=id_list)

# 4.유사도 검색

In [ ]:
query = '연봉 1억인 직장인의 소득세는 얼마인가요?'
retrieved_doc = tax_collection.query(query_texts=query, n_results=3)

In [ ]:
retrieved_doc['documents'][0]

# 5.llm 질의

In [ ]:
%pip install openai

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {'role': 'system', 'content': f'당신은 한국의 소득세 전문가 입니다. 아래 내용을 참고하여 사용자의 질문에 답변해주세요 {retrieved_doc["documents"][0]}'},
        {'role': 'user', 'content': query}
    ]
)

In [ ]:
response.choices[0].message.content